In [1]:
import sys
print(sys.executable)

C:\Users\anjal\Projects\attrition-analysis\.venv\Scripts\python.exe


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv')
print(df.shape)


(1470, 35)


In [3]:
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Age                       1470 non-null   int64
 1   Attrition                 1470 non-null   str  
 2   BusinessTravel            1470 non-null   str  
 3   DailyRate                 1470 non-null   int64
 4   Department                1470 non-null   str  
 5   DistanceFromHome          1470 non-null   int64
 6   Education                 1470 non-null   int64
 7   EducationField            1470 non-null   str  
 8   EmployeeCount             1470 non-null   int64
 9   EmployeeNumber            1470 non-null   int64
 10  EnvironmentSatisfaction   1470 non-null   int64
 11  Gender                    1470 non-null   str  
 12  HourlyRate                1470 non-null   int64
 13  JobInvolvement            1470 non-null   int64
 14  JobLevel                  1470 non-null   int64
 15

In [5]:
df.nunique().sort_values().head()

EmployeeCount    1
Over18           1
StandardHours    1
Attrition        2
OverTime         2
dtype: int64

In [6]:
constant_cols = [c for c in df.columns if df[c].nunique() == 1]
print("Dropping:", constant_cols)

df = df.drop(columns=constant_cols)
df['Attrition_Flag'] = (df['Attrition'] == 'Yes').astype(int)

print(df.shape)

Dropping: ['EmployeeCount', 'Over18', 'StandardHours']
(1470, 33)


In [7]:
df['Attrition'].value_counts(normalize=True)

Attrition
No     0.838776
Yes    0.161224
Name: proportion, dtype: float64

## Duplicate Rows Check

In [8]:
print("Fully duplicated rows:", df.duplicated().sum())

Fully duplicated rows: 0


## Identifier Columns

`EmployeeNumber` is a unique row identifier with no predictive value for attrition. It must be excluded from modeling features (kept only for joins/lookups), otherwise it risks leaking a proxy for row order/index into the model.

In [9]:
print("EmployeeNumber unique values:", df['EmployeeNumber'].nunique(), "/ rows:", len(df))
id_cols = ['EmployeeNumber']

EmployeeNumber unique values: 1470 / rows: 1470


## Feature Type Audit

Classify every remaining column as: target, identifier (excluded), categorical (object dtype), binary (2 unique values), or numerical. This drives the preprocessing pipeline built later in `03_modeling.ipynb`.

In [10]:
target_col = 'Attrition'  # raw Yes/No label; Attrition_Flag is its numeric mirror

feature_cols = [c for c in df.columns if c not in id_cols + [target_col, 'Attrition_Flag']]

# pandas 3.0 reports text columns as dtype "str", not "object" -> use is_numeric_dtype instead
categorical_cols = [c for c in feature_cols if not pd.api.types.is_numeric_dtype(df[c]) and df[c].nunique() > 2]
binary_cols = [c for c in feature_cols if df[c].nunique() == 2]
numerical_cols = [c for c in feature_cols if c not in categorical_cols and c not in binary_cols]

print(f"Categorical ({len(categorical_cols)}):", categorical_cols)
print(f"\nBinary ({len(binary_cols)}):", binary_cols)
print(f"\nNumerical ({len(numerical_cols)}):", numerical_cols)

Categorical (5): ['BusinessTravel', 'Department', 'EducationField', 'JobRole', 'MaritalStatus']

Binary (3): ['Gender', 'OverTime', 'PerformanceRating']

Numerical (22): ['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']


## Save Cleaned Dataset

Persist the cleaned dataframe (constant columns dropped, `Attrition_Flag` added) to `data/processed/` so downstream notebooks (`02_eda.ipynb`, `03_modeling.ipynb`) and the training script don't repeat this step.

In [11]:
import os

os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/cleaned_employee_attrition.csv', index=False)
print("Saved:", df.shape)

Saved: (1470, 33)
